# 🔧 SMD Dataset Preprocessor

**Dataset:** Server Machine Dataset (SMD)  
**Source:** [NetManAIOps/OmniAnomaly](https://github.com/NetManAIOps/OmniAnomaly)

Converts every raw `.txt` machine file into clean, labelled CSV files ready for ML.

---

### Output layout
```
data/processed/
├── train/          ← per-machine CSVs (38 features + machine_id)
├── test/           ← per-machine CSVs (38 features + label + machine_id)
├── all_train.csv   ← all machines combined
└── all_test.csv    ← all machines combined
```

### ML mapping
| Variable | Source | Notes |
|----------|--------|-------|
| `X_train` | `all_train.csv` — 38 feature cols | **Normal data only.** LSTM-AE is unsupervised — it learns what *normal* looks like, so **no label is needed** during training. |
| `X_test` | `all_test.csv` — 38 feature cols | Mixed normal + anomalous rows fed to the trained encoder → per-timestep reconstruction error. |
| `y_test` | `all_test.csv` — `label` col | Ground-truth **used only for evaluation** (F1, precision, recall). Never seen by the model. |

In [1]:
# ── 0. Imports & paths ────────────────────────────────────────────────────────
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from pathlib import Path
from IPython.display import display

# Project root is two levels up from notebooks/
PROJECT_ROOT = Path('../../').resolve()
DATA_DIR     = PROJECT_ROOT / 'data' / 'raw'  / 'ServerMachineDataset'
OUT_DIR      = PROJECT_ROOT / 'data' / 'processed'

print(f'Project root : {PROJECT_ROOT}')
print(f'Raw data     : {DATA_DIR}')
print(f'Output dir   : {OUT_DIR}')
print(f'Data exists  : {DATA_DIR.exists()}')

Project root : D:\ojt-project
Raw data     : D:\ojt-project\data\raw\ServerMachineDataset
Output dir   : D:\ojt-project\data\processed
Data exists  : True


In [2]:
# ── 1. Exact 38 SMD feature names (OmniAnomaly dataset README order) ──────────
FEATURE_NAMES = [
    'cpu_r',
    'load_1',
    'load_5',
    'load_15',
    'mem_shmem',
    'mem_u',
    'mem_u_e',
    'total_mem',
    'disk_q',
    'disk_r',
    'disk_rb',
    'disk_svc',
    'disk_u',
    'disk_w',
    'disk_wa',
    'disk_wb',
    'si',
    'so',
    'eth1_fi',
    'eth1_fo',
    'eth1_pi',
    'eth1_po',
    'tcp_tw',
    'tcp_use',
    'active_opens',
    'curr_estab',
    'in_errs',
    'in_segs',
    'listen_overflows',
    'out_rsts',
    'out_segs',
    'passive_opens',
    'retransegs',
    'tcp_timeouts',
    'udp_in_dg',
    'udp_out_dg',
    'udp_rcv_buf_errs',
    'udp_snd_buf_errs',
]

NUM_FEATURES = len(FEATURE_NAMES)
assert NUM_FEATURES == 38, f'Expected 38 features, got {NUM_FEATURES}'
print(f'Feature count verified: {NUM_FEATURES}')
print('Features:', FEATURE_NAMES)

Feature count verified: 38
Features: ['cpu_r', 'load_1', 'load_5', 'load_15', 'mem_shmem', 'mem_u', 'mem_u_e', 'total_mem', 'disk_q', 'disk_r', 'disk_rb', 'disk_svc', 'disk_u', 'disk_w', 'disk_wa', 'disk_wb', 'si', 'so', 'eth1_fi', 'eth1_fo', 'eth1_pi', 'eth1_po', 'tcp_tw', 'tcp_use', 'active_opens', 'curr_estab', 'in_errs', 'in_segs', 'listen_overflows', 'out_rsts', 'out_segs', 'passive_opens', 'retransegs', 'tcp_timeouts', 'udp_in_dg', 'udp_out_dg', 'udp_rcv_buf_errs', 'udp_snd_buf_errs']


In [3]:
# ── 2. Helper: load a raw .txt file with no header ────────────────────────────
def load_txt(path: Path, columns: list) -> pd.DataFrame:
    """
    Read a comma-separated .txt file with no header row and assign column names.
    Raises FileNotFoundError with a helpful message if the file is missing.
    """
    if not path.exists():
        raise FileNotFoundError(
            f'File not found: {path}\n'
            f'Run scripts/download_dataset.py first.'
        )
    return pd.read_csv(path, header=None, names=columns)


# ── 3. Validate the raw data directory ───────────────────────────────────────
train_dir = DATA_DIR / 'train'
if not train_dir.exists():
    raise RuntimeError(
        f'train/ directory not found at: {train_dir}\n'
        f'Run scripts/download_dataset.py first.'
    )

machine_files = sorted(train_dir.glob('*.txt'))
print(f'Machines found: {len(machine_files)}')
for mf in machine_files:
    print(f'   {mf.name}')

Machines found: 27
   machine-1-1.txt
   machine-1-2.txt
   machine-1-3.txt
   machine-1-4.txt
   machine-1-5.txt
   machine-1-6.txt
   machine-1-7.txt
   machine-1-8.txt
   machine-2-1.txt
   machine-2-2.txt
   machine-2-3.txt
   machine-2-4.txt
   machine-2-5.txt
   machine-2-6.txt
   machine-2-7.txt
   machine-2-8.txt
   machine-3-1.txt
   machine-3-10.txt
   machine-3-11.txt
   machine-3-2.txt
   machine-3-3.txt
   machine-3-4.txt
   machine-3-5.txt
   machine-3-6.txt
   machine-3-7.txt
   machine-3-8.txt
   machine-3-9.txt


In [4]:
# ── 4. Process every machine ─────────────────────────────────────────────────
#
# For each machine file:
#   A) Load train data  → 38 named feature columns
#   B) Load test data   → 38 named feature columns
#   C) Load test labels → verify len(labels) == len(test) before joining
#   D) Attach label (0=normal, 1=anomaly) and machine_id columns
#   E) Save per-machine CSVs to data/processed/train/ and data/processed/test/
#   F) Collect per-machine stats for the final summary table

all_train_frames = []
all_test_frames  = []
summary_rows     = []

for mf in machine_files:
    machine_id = mf.stem  # e.g. 'machine-1-1'

    train_path = DATA_DIR / 'train'      / mf.name
    test_path  = DATA_DIR / 'test'       / mf.name
    label_path = DATA_DIR / 'test_label' / mf.name

    # ── A) Load train ────────────────────────────────────────────────────────
    train_df = load_txt(train_path, FEATURE_NAMES)
    assert train_df.shape[1] == NUM_FEATURES, (
        f'{machine_id} train: expected {NUM_FEATURES} cols, got {train_df.shape[1]}'
    )

    # ── B) Load test ─────────────────────────────────────────────────────────
    test_df = load_txt(test_path, FEATURE_NAMES)
    assert test_df.shape[1] == NUM_FEATURES, (
        f'{machine_id} test: expected {NUM_FEATURES} cols, got {test_df.shape[1]}'
    )

    # ── C) Load labels & row-count validation ────────────────────────────────
    label_df = load_txt(label_path, ['label'])
    if len(label_df) != len(test_df):
        raise ValueError(
            f'{machine_id}: test rows ({len(test_df)}) != '
            f'label rows ({len(label_df)}). Files may be corrupted.'
        )

    # ── D) Attach metadata columns ───────────────────────────────────────────
    train_df['machine_id'] = machine_id
    test_df['label']       = label_df['label'].values   # 0 = normal, 1 = anomaly
    test_df['machine_id']  = machine_id

    # ── E) Save per-machine CSVs ─────────────────────────────────────────────
    train_out = OUT_DIR / 'train' / mf.name.replace('.txt', '.csv')
    test_out  = OUT_DIR / 'test'  / mf.name.replace('.txt', '.csv')
    train_out.parent.mkdir(parents=True, exist_ok=True)
    test_out.parent.mkdir(parents=True, exist_ok=True)
    train_df.to_csv(train_out, index=False)
    test_df.to_csv(test_out,  index=False)

    # ── F) Collect stats ─────────────────────────────────────────────────────
    n_anom   = int(test_df['label'].sum())
    anom_pct = n_anom / len(test_df) * 100

    all_train_frames.append(train_df)
    all_test_frames.append(test_df)
    summary_rows.append({
        'Machine':       machine_id,
        'Train Rows':    len(train_df),
        'Test Rows':     len(test_df),
        'Anomalies':     n_anom,
        'Anomaly (%)':   round(anom_pct, 3),
    })

    print(
        f'  {machine_id:<15}  '
        f'train={len(train_df):>6,}  '
        f'test={len(test_df):>6,}  '
        f'anomalies={n_anom:>5,}  ({anom_pct:.2f}%)'
    )

print('\nAll machines processed successfully.')

  machine-1-1      train=28,479  test=28,479  anomalies=2,694  (9.46%)


  machine-1-2      train=23,694  test=23,694  anomalies=  542  (2.29%)


  machine-1-3      train=23,702  test=23,703  anomalies=  817  (3.45%)


  machine-1-4      train=23,706  test=23,707  anomalies=  720  (3.04%)


  machine-1-5      train=23,705  test=23,706  anomalies=  100  (0.42%)


...

  machine-3-5      train=23,690  test=23,691  anomalies=  426  (1.80%)


  machine-3-6      train=28,726  test=28,726  anomalies=1,194  (4.16%)


  machine-3-7      train=28,705  test=28,705  anomalies=  434  (1.51%)


  machine-3-8      train=28,703  test=28,704  anomalies=1,371  (4.78%)


  machine-3-9      train=28,713  test=28,713  anomalies=  303  (1.06%)

All machines processed successfully.


In [5]:
# ── 5. Build combined datasets ────────────────────────────────────────────────
#
# all_train.csv  →  38 feature columns  + machine_id      (NO label)
# all_test.csv   →  38 feature columns  + label + machine_id
#
# Column order: features first, then metadata — mirrors the ML split:
#   X = df[FEATURE_NAMES]   y = df['label']

all_train = pd.concat(all_train_frames, ignore_index=True)
all_test  = pd.concat(all_test_frames,  ignore_index=True)

train_cols = FEATURE_NAMES + ['machine_id']
test_cols  = FEATURE_NAMES + ['label', 'machine_id']

all_train = all_train[train_cols]
all_test  = all_test[test_cols]

# Save combined files
OUT_DIR.mkdir(parents=True, exist_ok=True)
all_train.to_csv(OUT_DIR / 'all_train.csv', index=False)
all_test.to_csv(OUT_DIR  / 'all_test.csv',  index=False)

print(f'all_train.csv  saved  →  {len(all_train):,} rows  x  {len(train_cols)} columns')
print(f'all_test.csv   saved  →  {len(all_test):,} rows  x  {len(test_cols)} columns')
print(f'Output directory: {OUT_DIR}')

all_train.csv  saved  →  679,683 rows  x  39 columns
all_test.csv   saved  →  679,698 rows  x  40 columns
Output directory: D:\ojt-project\data\processed


In [6]:
# ── 6. Summary table ──────────────────────────────────────────────────────────
summary_df = pd.DataFrame(summary_rows)

# Totals row
totals = {
    'Machine':     'TOTAL',
    'Train Rows':  summary_df['Train Rows'].sum(),
    'Test Rows':   summary_df['Test Rows'].sum(),
    'Anomalies':   summary_df['Anomalies'].sum(),
    'Anomaly (%)': round(
        summary_df['Anomalies'].sum() / summary_df['Test Rows'].sum() * 100, 3
    ),
}
summary_full = pd.concat(
    [summary_df, pd.DataFrame([totals])], ignore_index=True
)

print('=== Per-Machine Summary ===')
display(summary_full.style
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#1e293b'), ('color', '#f8fafc'),
                                      ('font-size', '12px'), ('padding', '8px 14px'),
                                      ('text-align', 'center')]},
        {'selector': 'td', 'props': [('padding', '5px 14px'), ('font-size', '12px'),
                                      ('text-align', 'center')]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f1f5f9')]},
        {'selector': 'tr:last-child td',   'props': [('font-weight', 'bold'),
                                                       ('background-color', '#dbeafe')]},
    ])
    .format({
        'Train Rows':   '{:,}',
        'Test Rows':    '{:,}',
        'Anomalies':    '{:,}',
        'Anomaly (%)':  '{:.3f}',
    })
    .hide(axis='index')
    .bar(subset=['Anomaly (%)'], color='#fca5a5', vmin=0)
)

=== Per-Machine Summary ===


Machine,Train Rows,Test Rows,Anomalies,Anomaly (%)
machine-1-1,"28,479","28,479","2,694",9.460
machine-1-2,"23,694","23,694",542,2.287
machine-1-3,"23,702","23,703",817,3.447
machine-1-4,"23,706","23,707",720,3.037
machine-1-5,"23,705","23,706",100,0.422
machine-1-6,"23,688","23,689","3,708",15.653
machine-1-7,"23,697","23,697","2,398",10.119
machine-1-8,"23,698","23,699",763,3.220
machine-2-1,"23,693","23,694","1,170",4.938
machine-2-2,"23,699","23,700","2,833",11.954


In [7]:
# ── 7. Final high-level summary ───────────────────────────────────────────────
total_train  = all_train.shape[0]
total_test   = all_test.shape[0]
total_anom   = int(all_test['label'].sum())
anom_pct_all = total_anom / total_test * 100

final = pd.DataFrame([
    {
        'Dataset':      'Train  (all_train.csv)',
        'Rows':         total_train,
        'Features':     '38',
        'Anomalies':    'N/A',
        'Anomaly (%)':  'N/A',
        'ML Variable':  'X_train',
    },
    {
        'Dataset':      'Test   (all_test.csv)',
        'Rows':         total_test,
        'Features':     '38 + label',
        'Anomalies':    total_anom,
        'Anomaly (%)':  f'{anom_pct_all:.2f}%',
        'ML Variable':  'X_test  /  y_test',
    },
])

print('=== Dataset | Rows | Features | Anomalies | Anomaly % ===')
display(final.style
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#14532d'), ('color', '#f0fdf4'),
                                      ('font-size', '13px'), ('padding', '10px 16px')]},
        {'selector': 'td', 'props': [('padding', '8px 16px'), ('font-size', '13px')]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f0fdf4')]},
    ])
    .format({'Rows': '{:,}'}, na_rep='N/A')
    .hide(axis='index')
)

print()
print('Preprocessing complete!')
print(f'  all_train.csv : {total_train:,} rows')
print(f'  all_test.csv  : {total_test:,}  rows  ({total_anom:,} anomalies, {anom_pct_all:.2f}%)')

=== Dataset | Rows | Features | Anomalies | Anomaly % ===


Dataset,Rows,Features,Anomalies,Anomaly (%),ML Variable
Train (all_train.csv),"679,683",38,N/A,N/A,X_train
Test (all_test.csv),"679,698",38 + label,27689,4.07%,X_test / y_test



Preprocessing complete!
  all_train.csv : 679,683 rows
  all_test.csv  : 679,698  rows  (27,689 anomalies, 4.07%)


In [8]:
# ── 8. Quick sanity checks ────────────────────────────────────────────────────
print('=== Sanity Checks ===')

# Column names correct?
assert list(all_train.columns[:38]) == FEATURE_NAMES, 'Column name mismatch in all_train'
assert list(all_test.columns[:38])  == FEATURE_NAMES, 'Column name mismatch in all_test'
print('  Column names     : OK')

# No missing values?
assert all_train[FEATURE_NAMES].isnull().sum().sum() == 0, 'NaN in train features'
assert all_test[FEATURE_NAMES].isnull().sum().sum()  == 0, 'NaN in test features'
print('  Missing values   : None (OK)')

# Labels only 0 or 1?
label_vals = set(all_test['label'].unique())
assert label_vals <= {0, 1}, f'Unexpected label values: {label_vals}'
print(f'  Label values     : {sorted(label_vals)} (OK)')

# Feature values in expected range [0, 1]?
f_min = all_train[FEATURE_NAMES].min().min()
f_max = all_train[FEATURE_NAMES].max().max()
print(f'  Feature range    : [{f_min:.4f}, {f_max:.4f}]  (pre-normalised by dataset authors)')

# Per-machine CSVs exist?
n_train_csvs = len(list((OUT_DIR / 'train').glob('*.csv')))
n_test_csvs  = len(list((OUT_DIR / 'test').glob('*.csv')))
print(f'  Per-machine CSVs : {n_train_csvs} train  /  {n_test_csvs} test')

print()
print('All checks passed. Data is ready for model training.')

=== Sanity Checks ===
  Column names     : OK
  Missing values   : None (OK)
  Label values     : [np.int64(0), np.int64(1)] (OK)
  Feature range    : [0.0000, 1.0000]  (pre-normalised by dataset authors)
  Per-machine CSVs : 27 train  /  27 test

All checks passed. Data is ready for model training.
